# Figure 1A

### Setting up mining

In [ ]:
%%bash
### Downloading and splitting virus databases

# ### CHVD
# wget https://zenodo.org/records/4498884/files/CHVD_clustered_mash99_v1.tar.gz?download=1 -O chvd_mash99.tar.gz
# tar -xzf chvd_mash99.tar.gz
# seqkit split --by-size 10000 CHVD_clustered_mash99_v1.fasta --out-dir chvd_split -j 4

# ### CNGVC
# wget https://zenodo.org/records/14671177/files/cnGVC.vOTUs.fa.tar.gz?download=1 -O cnGVC.vOTUs.fa.tar.gz
# tar -xzf cnGVC.vOTUs.fa.tar.gz
# seqkit split --by-size 10000 cnGVC.fna --out-dir cngvd_split -j 4

# ### CNGVR
# wget https://download.cncb.ac.cn/OMIX/OMIX007646/OMIX007646-01.fa.gz
# seqkit split --by-size 10000 OMIX007646-01.fa.gz --out-dir cngvr_split -j 4

# ## MMGE
# wget https://mai.fudan.edu.cn/mgedb/client/file/all_mge_seq.zip
# unzip all_mge_seq.zip
# seqkit split --by-size 10000 all_mge_seq.fasta --out-dir mmge_split -j 4

# ### OPD
# wget https://ftp.cngb.org/pub/CNSA/data5/CNP0003685/CNS0634334/CNA0051948/opd-contigs.fa.gz
# seqkit split --by-size 10000 opd-contigs.fa.gz --out-dir opd_split -j 4

# ### OVD
# wget https://github.com/grc145/Temp/raw/refs/heads/master/OVD-genomes.fa.bz2
# seqkit split --by-size 10000 OVD-genomes.fa.bz2 --out-dir ovd_split -j 4

# ### SMGC
# wget https://ftp.ebi.ac.uk/pub/databases/metagenomics/genome_sets/skin_microbiome/03_finalcatalogue.tar.gz
# tar -xvf 03_finalcatalogue.tar.gz
# touch smgc_virus_list.txt
# for i in {623..7557}; do echo "03_finalcatalogue/SMGC_${i}.fa" >> smgc_virus_list.txt; done
# seqkit concat -X smgc_virus_list.txt --full -o SMGC_virus_all.fa
# seqkit split --by-size 10000 SMGC_virus_all.fa --out-dir smgc_split -j 4

# ### VMGC
# wget https://zenodo.org/records/10457006/files/VMGC_virus.tar.gz?download=1 -O VMGC_virus.tar.gz
# tar -xvf VMGC_virus.tar.gz
# seqkit split --by-size 10000 VMGC_virus_all.fa --out-dir vmgc_split -j 4

In [7]:
### Downloading and splitting human IMG/VR4 viruses
# metadata downloaded from: https://genome.jgi.doe.gov/portal/pages/dynamicOrganismDownload.jsf?organism=IMG_VR#
import polars as pl

imgvr_meta = pl.read_csv('IMGVR_all_Sequence_information.tsv',
    separator='\t', columns=['UVIG', 'Ecosystem classification'])

# identify human samples
(
    imgvr_meta
        .filter(
            (pl.col('Ecosystem classification').str.contains('Human'))
        )
        .group_by('Ecosystem classification')
        .len()
        .sort(pl.col('len'), descending=True)
        .write_csv('IMGVR_human_ecosystem_classification_counts.csv')
)

imgvr_human_uvigs = (
    imgvr_meta
        .filter(
            (pl.col('Ecosystem classification').str.contains('Human'))
        )
)
# imgvr_human_uvigs[['UVIG']].write_csv('IMGVR_human_uvigs.csv')

# extract human-associated sequences
# !seqkit grep --pattern-file IMGVR_human_uvigs.csv IMGVR4_SEQUENCES.fna --id-regexp "^(.*?)\|" -j 4 --out-file IMGVR4_human_sequences.fna.gz
# !seqkit split --by-size 10000 IMGVR4_human_sequences.fna.gz --out-dir IMGVR4_human_sequences_split -j 4

In [ ]:
### Virus database sequence counts
# CHVD: 93,860
# CNGVC: 93,462
# CNGVR: 120,568
# MMGE: 517,251
# OPD: 189,859
# OVD: 48,425
# SMGC: 6,935
# VMGC: 14,224
# IMG/VR4: 708,593

# TOTAL: 1,793,177

In [ ]:
### ENA human airways, skin, and urogenital samples

# download ena metagenome metadata
# !curl -X 'POST' \
# 'https://www.ebi.ac.uk/ena/portal/api/search' -H 'accept: */*' -H 'Content-Type: application/x-www-form-urlencoded' \
# -d 'excludeAccessionType=&download=false&query=&excludeAccessions=&includeMetagenomes=true&dataPortal=metagenome&searchCurations=false&includeAccessionType=&includeAccessions=&format=tsv&fields=analysis_accession%2Csample_accession%2Crun_accession%2Cscientific_name%2Cbroad_scale_environmental_context%2Cenvironment_biome%2Chost_body_site%2Cgenerated_ftp%2Cfirst_public&dccDataOnly=false&rule=&result=analysis' \
# > ena_metagenomes_2025_08_29.tsv

# load ena metagenome metadata
import polars as pl
ena_meta = pl.read_csv('ena_metagenomes_2025_08_29.tsv', separator='\t')

# manually look through top organisms to find airways, skin, and urogenital samples
ena_meta.group_by('scientific_name').len().sort('len', descending=True).write_csv('ena_metagenomes_top_organisms.csv')

# create a samplesheet of airways, skin, and urogenital samples
ena_human_samples = (
    ena_meta
        .filter(
            (pl.col('scientific_name').str.contains('oral')) |
            (pl.col('scientific_name').str.contains('nasopharyngeal')) |
            (pl.col('scientific_name').str.contains('lung')) |
            (pl.col('scientific_name').str.contains('saliva ')) |
            (pl.col('scientific_name').str.contains('skin')) |
            (pl.col('scientific_name').str.contains('vaginal ')) |
            (pl.col('scientific_name').str.contains('urinary'))
        )
)

(
    ena_human_samples
        .rename({
            'analysis_accession':'sample',
            'sample_accession':'biosample',
            'run_accession':'acc'
        })
        .with_columns([
            pl.lit('ENA').alias('source_db'),
            pl.lit('r2025_09').alias('release'),
            (pl.lit('https://') + pl.col('generated_ftp')).alias('fasta'),
        ])
        .drop('generated_ftp')
        .write_csv('ena_human_samplesheet.csv')
)

# count number of ENA samples
print(f"Number of human-associated metagenomes: {ena_human_samples.height}")

# compress output files
# !gzip ena_human_samplesheet.csv ena_metagenomes_top_organisms.csv

In [ ]:
### Logan human airways, skin, and urogenital samples

# parse SRA metadata to find human-associated metagenomes
# import duckdb

# duckdb.sql('''
# INSTALL httpfs;
# LOAD httpfs;
# INSTALL parquet;
# LOAD parquet;
# COPY (
#   SELECT 
#     acc, assay_type, consent, libraryselection, librarysource, organism
#   FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
#   WHERE assay_type != 'AMPLICON'
#   AND consent = 'public'
#   AND libraryselection != 'PCR'
#   AND (librarysource = 'METAGENOMIC'
#     OR organism LIKE '%microbiom%'
#     OR organism LIKE '%metagenom%')
# ) TO '2025_08_29_sra_metadata.parquet' (FORMAT 'parquet');
# ''')

import polars as pl

# identify top organisms
sra_meta = (
    pl.read_parquet('2025_05_29_sra_metadata.parquet',
        columns=['acc', 'organism']
    )
)

# write out file of most prevalent organisms
(
    sra_meta
        .group_by('organism')
        .len()
        .sort('len', descending=True)
        .write_csv('sra_metagenomes_top_organisms.csv')
)

# identify human-associated metagenomes
human_organisms = set([
    'human oral metagenome',
    'human nasopharyngeal metagenome',
    'oral metagenome',
    'human lung metagenome',
    'respiratory tract metagenome',
    'human saliva metagenome',
    'upper respiratory tract metagenome',
    'human sputum metagenome',
    'lung metagenome',
    'human tracheal metagenome',
    'oral-nasopharyngeal metagenome',
    'human skin metagenome',
    'skin metagenome',
    'human vaginal metagenome',
    'vaginal metagenome',
    'human urinary tract metagenome',
    'urinary tract metagenome',
    'human reproductive system metagenome',
    'reproductive system metagenome'
])

sra_meta_human = (
    sra_meta
        .filter(
            (pl.col('organism').is_in(human_organisms))
        )
)

# identify samples in logan
# !wget https://s3.amazonaws.com/logan-pub/stats/logan-seqstats-contigs-v1.1.parquet
logan_stats = pl.read_parquet('logan-seqstats-contigs-v1.1.parquet')
sra_meta_human_logan = (
    sra_meta_human
        .filter(pl.col('acc').is_in(logan_stats['accession']))
)
print(f"Number of human-associated metagenomes in Logan: {sra_meta_human_logan.height}")

# create samplesheet for mining logan metagenomes
(
    sra_meta_human_logan
        .with_columns([
            pl.lit('LOGAN').alias('source_db'),
            pl.lit('r2025_09').alias('release'),
            (pl.lit('s3://logan-pub/c/') + pl.col('acc') + '/' + pl.col('acc') + '.contigs.fa.zst').alias('fasta'),
            pl.lit(None).alias('tar')
        ])
        .write_csv('logan_human_samplesheet.csv')
)

# compress output files
# !gzip logan_human_samplesheet.csv sra_metagenomes_top_organisms.csv

Number of human-associated metagenomes in Logan: 54908


In [3]:
### SPIRE human airways, skin, and urogenital samples

import polars as pl

# load spire microontoloy
spire_ont = (
    pl.read_csv("https://swifter.embl.de/~fullam/spire/metadata/spire_v1_microntology.tsv.gz", separator="\t", has_header=False)
    .with_columns(
        pl.col("column_1").str.splitn(by=" ", n=3)
        .struct.rename_fields(["biosample", "bioproject", "ontology"])
        .alias("fields")
    ).unnest("fields")
)

# identify most common ontologies
(
    spire_ont
        .filter(pl.col("ontology").str.contains("animal host"))
        .group_by("ontology")
        .len()
        .sort('len', descending=True)
        .write_csv("spire_animal_host_ontologies.csv")
)

# filter to human airways, skin, and urogenital samples
spire_human = (
    spire_ont
        .filter(
            (pl.col("ontology").str.contains("animal host")) &
            (
                (pl.col("ontology").str.contains("mouth")) |
                (pl.col("ontology").str.contains("airway")) |
                (pl.col("ontology").str.contains("urogenital")) |
                (pl.col("ontology").str.contains("skin"))
            )
        )
        .unique('biosample')
)

# create samplesheet
(
    spire_human
        .with_columns([
            pl.lit(None).alias('acc'),
            pl.lit('SPIRE').alias('source_db'),
            pl.lit('r2025_09').alias('release'),
            (pl.lit('https://spire.embl.de/download_assembly/') + pl.col('biosample')).alias('fasta'),
            pl.lit(None).alias('tar'),
            pl.col('biosample').alias('sample'),
        ])
        .drop('bioproject', 'ontology', 'column_1')
).write_csv('spire_human_samplesheet.csv')

# count number of SPIRE samples
print(f"Number of human-associated metagenomes in SPIRE: {spire_human.height}")
# compress output files
# !gzip spire_human_samplesheet.csv spire_animal_host_ontologies.csv

Number of human-associated metagenomes in SPIRE: 9620


In [4]:
### Total assemblies = 79,131

In [ ]:
### All sequences were then run through UHVDB/mine using their respective samplesheets

### Mining summary

In [ ]:
%%bash
### Combine mining reports into one file

# echo -e "seq_name\ttopology\tcoordinates\tn_genes\tgenetic_code\tvirus_score\tfdr\tn_hallmarks\tmarker_enrichment\ttaxonomy\tgenome\tplasmid_hallmarks\tconj_genes\tzot_genes\tinoviridae_marker_genes\ttotal_markers\tn_dup_markers\tmarker_duplicity\tcontig_length\tprovirus\tproviral_length\tviral_genes\thost_genes\tcompleteness\tcompleteness_method\tkmer_freq\twarningsPrediction\tScore\tPfam\thits\tictv_family\tgenomad_virus_score_95\tviralverify_virus_score_15\tvirus_hallmarks_gt_3\tictv_known_family\tzot_gene\tinoviridae_marker_genes_gt_5\tviralverify_chromosome_plasmid\tcheckv_host_genes_gt_2\tplasmid_hallmarks_gt_0\tconj_genes_gt_0\tmarker_duplicity_penalty\tviral_score_sum\tuhvdb_virus_classification\tcompleteness_2\tcompleteness_method_2\tcheckv_quality\twarnings_2\taai_expected_length\taai_num_hits\taai_top_hit\taai_id\taai_af\tsource_db" \
#     > viruses.csvtk_concat.tsv

# for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/*/*/uhvdb/minereport/*.mine_summary.tsv.gz; do
#     zcat "$file" | tail -n +2 >> viruses.csvtk_concat.tsv
# done

# # identify all fasta IDs
# zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/r2025_0*/*/uhvdb/virusfilter/*.uhvdb_viruses.fna.gz \
#     | grep "^>" > uhpdb_virusfilter_ids.txt

In [9]:
### summarize number of viruses mined for UHVDB
import polars as pl

# read sequence ids
uhvdb_ids = pl.read_csv('uhvdb_virusfilter_ids.txt', has_header=False, new_columns=['seq_header'])

# read mine report
mine_report = pl.read_csv('viruses.csvtk_concat.tsv', separator='\t')

# create a set of sequence ids from uhvdb
uhvdb_seq_ids = set(
    uhvdb_ids
        .with_columns([
            pl.col('seq_header').str.replace('^>', '').str.split(' ').list[0].alias('seq_name')
        ])['seq_name']
)

# filter mine report to only those in uhvdb
mine_report_uhvdb_viruses = (
    mine_report
        .filter(
            (pl.col('seq_name').is_in(uhvdb_seq_ids)) &
            (
                (~pl.col('warnings_2').str.contains('>1.5x')) |
                (pl.col('warnings_2').is_null())
            )
        )
        .unique('seq_name')
)
print("Number of virus mined for UHVDB:", mine_report_uhvdb_viruses.height)

Number of virus mined for UHVDB: 3078958


In [10]:
### Determine number of confident/uncertain viruses from UHVDB mining
mine_report_uhvdb_viruses.group_by('uhvdb_virus_classification').len().sort('len', descending=True)

uhvdb_virus_classification,len
str,u32
"""uncertain""",1979132
"""confident""",1099826


In [11]:
### Determine number of high-quality viruses from UHVDB mining
# filter mine report to only those in uhvdb
mine_report_uhvdb_hq = (
    mine_report
        .filter(
            (pl.col('seq_name').is_in(uhvdb_seq_ids)) &
            (
                (~pl.col('warnings_2').str.contains('>1.5x')) |
                (pl.col('warnings_2').is_null())
            )
        )
        .filter((pl.col('completeness_2') != 'NA') & (pl.col('completeness_2').is_not_null()))
        .with_columns([pl.col('completeness_2').cast(pl.Float64)])
        .filter(
            (pl.col('completeness_2') >= 90)
        )
        .unique('seq_name')
)
print("Number of HQ virus sequences in UHVDB:", mine_report_uhvdb_hq.height)

Number of HQ virus sequences in UHVDB: 562711


### Combine UHGV with new HQ viruses

In [ ]:
%%bash
### Dereplicate UHVDB HQ viruses

# # combine all newly mined HQ viruses
# touch hq_viruses_cat.fasta.gz

# for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/*/*/uhvdb/hqfilter/*.hq_viruses.fna.gz; do
#     cat $file >> hq_viruses_cat.fasta.gz
# done

# # trim DTRs on new viruses
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/tr-trimmer \
#     hq_viruses_cat.fasta.gz \
#     --min-length 20 \
#     --include-tr-info \
#     > hq_viruses.tr-trimmer.fna

# # dereplicate new viruses
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/seq-hasher \
#     hq_viruses.tr-trimmer.fna \
#     --multi-kmer-hashing \
#     --circular-kmers \
#     > hq_viruses.seq-hasher.tsv

In [ ]:
### Identify uncertain UHGV viruses
# # download UHGV metadata
#!wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv

# # identify HQ uncertain viruses
# import polars as pl

# (
#     pl.read_csv('uhgv_metadata.tsv', separator='\t', ignore_errors=True)
#         .filter(
#             (pl.col('checkv_completeness') >= 90) &
#             (pl.col('viral_confidence') == 'Uncertain')
#         )
#         .write_csv('uhgv_hq_plus_uncertain.tsv', separator='\t', include_header=False)
# )

In [ ]:
%%bash
### Combine UHGV HQ uncertain viruses with HQ confident viruses & dereplicate

# # identify UHGV HQ uncertain
# wget https://portal.nersc.gov/cfs/m342/UHGV/genome_catalogs/uhgv_full.fna.gz

# seqkit grep \
#     uhgv_full.fna.gz \
#     --pattern-file uhgv_hq_plus_uncertain.tsv \
#     --out-file uhgv_hq_plus_uncertain.fna.gz

# # combine HQ uncertain with HQ confident
# wget https://portal.nersc.gov/cfs/m342/UHGV/genome_catalogs/uhgv_hq_plus.fna.gz

# cat uhgv_hq_plus.fna.gz \
#     uhgv_hq_plus_uncertain.fna.gz \
#     > uhgv_hq_plus_confident_w_uncertain.fna.gz

# # run seqhasher to dereplicate genomes
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/seq-hasher \
#     uhgv_hq_plus_confident_w_uncertain.fna.gz \
#     --multi-kmer-hashing \
#     --circular-kmers \
#     > uhgv.seq-hasher.tsv

In [ ]:
%%bash
### Combine UHGV with HQ viruses to create UHVDB r2025_09

# # combine UHGV and new virus seq-hasher outputs to identify unique hashes
# cat uhgv.seq-hasher.tsv \
#     hq_viruses.seq-hasher.tsv \
#     > combined.seq-hasher.tsv

# csvtk uniq \
#     combined.seq-hasher.tsv \
#     --no-header-row \
#     --fields 2 \
#     --tabs \
#     --out-file hq_viruses.csvtk_uniq.tsv.gz

# # extract seq ids
# csvtk cut \
#     --tabs \
#     --fields 1 \
#     hq_viruses.csvtk_uniq.tsv.gz \
#     --out-file hq_viruses.unique_seq_ids.tsv

# # extract unique sequences from UHGV
# seqkit \
#     grep \
#     --pattern-file hq_viruses.unique_seq_ids.tsv \
#     uhgv_hq_plus_confident_w_uncertain.fna.gz \
#     -o uhgv.seqkit_derep.fna.gz

# # extract unique sequences from new viruses
# seqkit \
#     grep \
#     --pattern-file hq_viruses.unique_seq_ids.tsv \
#     hq_viruses.tr-trimmer.fna \
#     -o hq_viruses.seqkit_derep.fna.gz

# combine dereplicated UHGV and new viruses to create UHVDB
# cat uhgv.seqkit_derep.fna.gz \
#     hq_viruses.seqkit_derep.fna.gz \
#     > uhvdb_unique.fna.gz

In [ ]:
%%bash
# count the number of unique HQ sequences
cat hq_viruses.unique_seq_ids.tsv | wc -l
# 591,860 sequences with unique hashes

### Dereplicate UHVDB at genomovar level

In [ ]:
%%bash
### Cluster UHVDB r2025_09 at genomovar (99.5% ANI and 100% AF) level

# vclust \
#     prefilter \
#     --in uhvdb_unique.fna.gz \
#     --out uhvdb.vclust_derep_prefilter.txt \
#     --threads 36 \
#     --min-ident 0.99

# vclust \
#     align \
#     --in uhvdb_unique.fna.gz \
#     --out uhvdb.vclust_genomovars_ani.tsv \
#     --filter uhvdb.vclust_derep_prefilter.txt \
#     --filter-threshold 0.99 \
#     --threads 36 \
#     --out-ani 0.995 \
#     --out-qcov 1.0

# vclust \
#     cluster \
#     --in uhvdb.vclust_genomovars_ani.tsv \
#     --ids uhvdb.vclust_genomovars_ani.ids.tsv \
#     --out uhvdb.vclust_genomovars_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 0.995 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     uhvdb.vclust_genomovars_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file uhvdb.vclust_genomovars_reps.tsv

# seqkit \
#     grep \
#     uhvdb_unique.fna.gz \
#     --pattern-file uhvdb.vclust_genomovars_reps.tsv \
#     --out-file uhvdb.vclust_genomovars_reps.fna.gz

In [12]:
### Count number of genomovars in UHVDB
import polars as pl
uhvdb_derep_ids = set(pl.read_csv('uhvdb.vclust_genomovars_reps.tsv')['cluster'])
print("Number of HQ genomovars in UHVDB:", len(uhvdb_derep_ids))

# 458,321

Number of HQ genomovars in UHVDB: 458321


### Remove uncertain viruses with insufficient HMM hits

In [15]:
### Identify confident and uncertain viruses in UHVDB
import polars as pl

uhvdb_genomovars = set(
    pl.read_csv('uhvdb.vclust_genomovars_reps.tsv', has_header=False, new_columns=['seq_name'])
        ['seq_name']
)

# download UHGV metadata
# !wget https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv

uhgv_hq_genomovars = (
    pl.read_csv("uhgv_metadata.tsv", separator="\t",
        columns=['uhgv_genome', 'viral_confidence', 'genomad_virus_hallmarks', 'genomad_plasmid_hallmarks', 'genomad_virus_score'], ignore_errors=True)
        .filter(pl.col('uhgv_genome').is_in(uhvdb_genomovars))
        .rename({'uhgv_genome': 'seq_name'})
)
print("Number of UHGV HQ genomovars in UHVDB:", uhgv_hq_genomovars.height)

# filter mine report to only those in unique seq ids
uhvdb_hq_genomovars = (
    pl.read_csv('viruses.csvtk_concat.tsv', separator='\t', columns=['seq_name', 'uhvdb_virus_classification', 'n_hallmarks', 'plasmid_hallmarks', 'virus_score'])
        .filter(pl.col('seq_name').is_in(uhvdb_genomovars))
        .unique('seq_name')
        .rename({'uhvdb_virus_classification': 'viral_confidence', 'n_hallmarks': 'genomad_virus_hallmarks', 'plasmid_hallmarks': 'genomad_plasmid_hallmarks', 'virus_score': 'genomad_virus_score'})
        .with_columns([
            pl.col('viral_confidence').str.replace('confident', 'Confident')
        ])
        .with_columns([
            pl.col('viral_confidence').str.replace('uncertain', 'Uncertain'),
        ])
)
print("Number of new dereplicated HQ genomovarss in UHVDB:", uhvdb_hq_genomovars.height)

# combine UHVDB and UHGV metadata
uhgv_uhvdb_hq_genomovars_concat = pl.concat([
    uhgv_hq_genomovars[['seq_name', 'viral_confidence', 'genomad_virus_hallmarks', 'genomad_plasmid_hallmarks', 'genomad_virus_score']],
    uhvdb_hq_genomovars[['seq_name', 'viral_confidence', 'genomad_virus_hallmarks', 'genomad_plasmid_hallmarks', 'genomad_virus_score']]
    ]).unique('seq_name')

# count uncertain vs confident genomes
print(uhgv_uhvdb_hq_genomovars_concat.group_by('viral_confidence').len())

# create pattern file of uncertain genomes
# (
#     uhgv_uhvdb_hq_genomovars_concat
#         .filter(pl.col('viral_confidence') == 'Uncertain')[['seq_name']]
#         .write_csv('uhvdb_hq_genomovars_uncertain.txt', separator='\t')
# )

Number of UHGV HQ genomovars in UHVDB: 137464
Number of new dereplicated HQ genomovarss in UHVDB: 320857
shape: (2, 2)
┌──────────────────┬────────┐
│ viral_confidence ┆ len    │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ Uncertain        ┆ 21352  │
│ Confident        ┆ 436969 │
└──────────────────┴────────┘


In [ ]:
%%bash
### Create fasta file of uncertain genomes and split
# # filter fasta file to uncertain genomes
# seqkit grep \
#     uhvdb.vclust_genomovars_reps.fna.gz \
#     --pattern-file uhvdb_hq_genomovars_uncertain.txt \
#     --out-file uhvdb_hq_genomovars_uncertain.fna

# # filter fasta file to confident genomes
# seqkit grep \
#     uhvdb.vclust_genomovars_reps.fna.gz \
#     --invert-match \
#     --pattern-file uhvdb_hq_genomovars_uncertain.txt \
#     --out-file uhvdb_hq_genomovars_confident.fna

# # split uncertain fasta into splits with 1,000 sequences each
# seqkit split2 \
#     uhvdb_hq_genomovars_uncertain.fna \
#     --by-size 1000 \
#     --out-dir uhvdb_hq_genomovars_uncertain_split

In [ ]:
### Setup genomad HMM search
# !wget https://zenodo.org/records/14886553/files/genomad_metadata_v1.9.tsv.gz?download=1 -O genomad_metadata_v1.9.tsv.gz

import polars as pl

# identify virus/plasmid hallmarks
genomad_hallmarks = (
    pl.read_csv('genomad_metadata_v1.9.tsv.gz', separator='\t', ignore_errors=True)
        .filter(
            (pl.col('PLASMID_HALLMARK') == 1) | 
            (pl.col('VIRUS_HALLMARK') == 1)
        )[['MARKER']]
        .with_columns([
            ("genomad_hmm_v1.9/" + pl.col('MARKER') + ".hmm").alias('hmm_path')
        ])[['hmm_path']]
        .write_csv('genomad_hallmarks_hmms.txt', include_header=False)
)

In [ ]:
%%bash
### Download genomad HMMs and combine hallmarks into one file

# wget https://zenodo.org/records/14886553/files/genomad_hmm_v1.9.tar.gz?download=1 -O genomad_hmm_v1.9.tar.gz
# # gunzip tar file
# gunzip genomad_hmm_v1.9.tar.gz

# # change to dir to store hmms
# tar -xvf genomad_hmm_v1.9.tar --files-from genomad_hallmarks_hmms.txt

# combine all hmms into one file
# cat genomad_hmm_v1.9/*.hmm > genomad_1_9_hallmarks.hmm

In [ ]:
%%bash
### Run genomad hallmark HMMsearch

# sbatch hallmark_hmmsearch.sh

In [ ]:
### Parese HMM search results
import polars as pl
import glob

# count hmm hallmark hits per genome
genomad_virus_hallmarks = set(
    pl.read_csv('genomad_metadata_v1.9.tsv.gz', separator='\t', ignore_errors=True)
    .filter(
        (pl.col('VIRUS_HALLMARK') == 1)
    )['MARKER']
)

genomad_plasmid_hallmarks = set(
    pl.read_csv('genomad_metadata_v1.9.tsv.gz', separator='\t', ignore_errors=True)
    .filter(
        (pl.col('PLASMID_HALLMARK') == 1)
    )['MARKER']
)

results = []
for file in glob.glob('hmm_results/*.tbl'):
    with open(file, 'r') as tbl:
        for line in tbl:
            if '#' in line[0]:
                continue
            strip_split = line.strip().split()
            protein = strip_split[0]
            genome = protein.rsplit('_', 1)[0]
            target = strip_split[2]
            results.append({'genome': genome, 'protein': protein, 'hallmark': target, 'evalue': float(strip_split[4])})
    tbl.close()

# convert dict to df
uncertain_hmm_hallmarks = (
    pl.DataFrame(results)
        .with_columns([
            pl.when(pl.col('hallmark').is_in(genomad_virus_hallmarks)).then(1).otherwise(0).alias('virus_hallmarks'),
            pl.when(pl.col('hallmark').is_in(genomad_plasmid_hallmarks)).then(1).otherwise(0).alias('plasmid_hallmarks'),
        ])
        .sort('evalue', descending=False)
        .group_by('protein')
        .first()
        .group_by(['genome'])
        .agg([pl.col('virus_hallmarks').sum().alias('virus_hallmarks'), pl.col('plasmid_hallmarks').sum().alias('plasmid_hallmarks')])
)

# write out genomes with >= 3 virus hallmarks and 0 plasmid hallmarks (would add 1 point to score)
uncertain2confident = (
    uncertain_hmm_hallmarks
        .filter(
            (pl.col('virus_hallmarks') >= 3) &
            (pl.col('plasmid_hallmarks') == 0)
        )
)

print("Number of sequences going from Uncertain to Confident:", uncertain2confident.shape[0])
# 8,621

uncertain2confident[['genome']].write_csv('uhvdb_hq_genomovars_uncertain2confident.txt', include_header=False)

In [ ]:
%%bash
### Create final UHVDB r2025_09 HQ HC fasta file

# # extract uncertain2confident genomes
# seqkit grep \
#     uhvdb_hq_genomovars_uncertain.fna \
#     --pattern-file uhvdb_hq_genomovars_uncertain2confident.txt \
#     --out-file uhvdb_uncertain2confident.fna

# # combine confident and new confident genomes
# cat uhvdb_hq_genomovars_confident.fna uhvdb_uncertain2confident.fna > uhvdb_hq_hc_genomovars.fna

# # count number of sequences in final UHVDB r2025_09 HQ HC fasta file
# grep -c "^>" uhvdb_hq_hc_genomovars.fna

# 445,590